In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
rfm = pd.read_csv("../data/notebook_dataset/rfm_table.csv", index_col=0)

rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,325,17,155164.66
12347.0,1,8,4921.53
12348.0,74,5,2019.40
12349.0,18,5,4452.84
12350.0,309,1,334.40


In [3]:
threshold = rfm['Monetary'].quantile(0.75)

rfm['HighValueCustomer'] = (rfm['Monetary'] >= threshold).astype(int)

In [4]:
rfm.head()

,Recency,Frequency,Monetary,HighValueCustomer
Customer ID,,,,
12346.0,325,17,155164.66,1
12347.0,1,8,4921.53,1
12348.0,74,5,2019.40,0
12349.0,18,5,4452.84,1
12350.0,309,1,334.40,0


In [27]:
X = rfm[['Recency','Frequency','Cluster']]

y = rfm['HighValueCustomer']

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
models = {

    "Logistic Regression": LogisticRegression(),

    "Decision Tree": DecisionTreeClassifier(),

    "Random Forest": RandomForestClassifier(),

    "Gradient Boosting": GradientBoostingClassifier(),

    "AdaBoost": AdaBoostClassifier(),

    "KNN": KNeighborsClassifier(),

    "Naive Bayes": GaussianNB(),

    "XGBoost": XGBClassifier(eval_metric="logloss"),

    "LightGBM": LGBMClassifier()
}

In [31]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


def evaluate_models(X_train, y_train, X_test, y_test, models):
    
    report = {}

    for name, model in models.items():
        
        # Train
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Probabilities for ROC-AUC
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:,1]
        else:
            y_prob = None

        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        if y_prob is not None:
            roc_auc = roc_auc_score(y_test, y_prob)
        else:
            roc_auc = None

        report[name] = {
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1,
            "ROC AUC": roc_auc
        }

        print(f"\n{name} Results")
        print("Accuracy:", accuracy)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1 Score:", f1)
        print("ROC AUC:", roc_auc)

    return report

In [32]:
report = evaluate_models(
    X_train,
    y_train,
    X_test,
    y_test,
    models
)


Logistic Regression Results
Accuracy: 0.8915054667788057
Precision: 0.8097165991902834
Recall: 0.7092198581560284
F1 Score: 0.7561436672967864
ROC AUC: 0.9385473112982555

Decision Tree Results
Accuracy: 0.8881412952060556
Precision: 0.7898832684824902
Recall: 0.7198581560283688
F1 Score: 0.7532467532467533
ROC AUC: 0.8543382048214438

Random Forest Results
Accuracy: 0.8915054667788057
Precision: 0.7886792452830189
Recall: 0.7411347517730497
F1 Score: 0.7641681901279708
ROC AUC: 0.9253872559368818

Gradient Boosting Results
Accuracy: 0.9007569386038689
Precision: 0.8306451612903226
Recall: 0.7304964539007093
F1 Score: 0.7773584905660378
ROC AUC: 0.9530073424194797

AdaBoost Results
Accuracy: 0.8839360807401178
Precision: 0.7432432432432432
Recall: 0.7801418439716312
F1 Score: 0.7612456747404844
ROC AUC: 0.9462709266774574

KNN Results
Accuracy: 0.8973927670311186
Precision: 0.8174603174603174
Recall: 0.7304964539007093
F1 Score: 0.7715355805243446
ROC AUC: 0.9083409572513235

Naive Ba

c:\Users\chinm\anaconda3\envs\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\chinm\anaconda3\envs\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [33]:
results_df = pd.DataFrame(report).T

results_df

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.891505,0.809717,0.709220,0.756144,0.938547
Decision Tree,0.888141,0.789883,0.719858,0.753247,0.854338
Random Forest,0.891505,0.788679,0.741135,0.764168,0.925387
Gradient Boosting,0.900757,0.830645,0.730496,0.777358,0.953007
AdaBoost,0.883936,0.743243,0.780142,0.761246,0.946271
KNN,0.897393,0.817460,0.730496,0.771536,0.908341
Naive Bayes,0.894029,0.822314,0.705674,0.759542,0.921220
XGBoost,0.897393,0.822581,0.723404,0.769811,0.951811
LightGBM,0.899075,0.816406,0.741135,0.776952,0.950748


In [25]:
rfm_log = pd.read_csv("../data/notebook_dataset/rfm_log_table.csv", index_col=0)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_log)

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=10,
    random_state=42
)

kmeans.fit(rfm_scaled)

rfm['Cluster'] = kmeans.labels_

rfm.to_csv("data/notebook_dataset/rfm_table_with_clusters.csv")

In [26]:
rfm = pd.read_csv("data/notebook_dataset/rfm_table_with_clusters.csv", index_col=0)

In [34]:
rfm.head()

,Recency,Frequency,Monetary,HighValueCustomer,Cluster
Customer ID,,,,,
12346.0,325,17,155164.66,1,1
12347.0,1,8,4921.53,1,1
12348.0,74,5,2019.40,0,2
12349.0,18,5,4452.84,1,2
12350.0,309,1,334.40,0,0
